In [4]:
import pandas as pd
from tqdm import tqdm
import os
from rdkit import Chem
from posebusters.modules.rmsd import check_rmsd

In [5]:
DF = pd.read_csv('2024_06_19_after_structure_filters.csv')#[:1000]
name = '2024_06_19_after_structure_filters'
CODES = [f'{row[1]['pdb_id']}_{row[1]["chain_id"]}_{row[1]["res_id"]}' for row in DF.iterrows()]

In [6]:
problematic = []

for code in tqdm(CODES):
    pid = code.split('_')[0]
    chain = code.split('_')[1]
    res = code.split('_')[2]
    if not os.path.exists(f'{name}/{pid}'):
        os.makedirs(f'{name}/{pid}')
    for i in range(0, 5):
        if os.path.exists(f'/vols/opig/users/durant/pose_classification/conformer_random_walks/new_pdb_dataset/{code}/{code}_ligand_{i}.sdf'):
            if not os.path.exists(f'{name}/{pid}/{code}_ligand_{i}.sdf'):
                os.system(f'cp /vols/opig/users/durant/pose_classification/conformer_random_walks/new_pdb_dataset/{code}/{code}_ligand_{i}.sdf {name}/{pid}/{code}_ligand_{i}.sdf')
        else:
            if code not in problematic:
                problematic.append(code)
    if os.path.exists(f'/vols/opig/users/durant/pose_classification/data_prep/new_pdb_dataset/2024_06_19_after_structure_filters_complexes/{pid}/{pid}_protein_cleaned.pdb'):
        if not os.path.exists(f'{name}/{pid}/{pid}_protein_cleaned.pdb'):
            os.system(f'cp /vols/opig/users/durant/pose_classification/data_prep/new_pdb_dataset/2024_06_19_after_structure_filters_complexes/{pid}/{pid}_protein_cleaned.pdb {name}/{pid}/{pid}_protein_cleaned.pdb')

print(problematic)

 48%|████▊     | 5048/10623 [28:44<35:41,  2.60it/s]  

In [11]:
print(len(problematic))

51


In [12]:
for code in tqdm(CODES):
    pid = code.split('_')[0]
    chain = code.split('_')[1]
    res = code.split('_')[2]
    if not os.path.exists(f'{name}/{pid}'):
        os.makedirs(f'{name}/{pid}')
    for num, i in enumerate(['inter_pl_clashes', 'internal_clashes', 'volume_overlap', 'energy_ratio']):
        if os.path.exists(f'/vols/opig/users/durant/pose_classification/conformer_random_walks/new_pdb_dataset_{i}/{pid}_{chain}/{pid}_{chain}_ligand_0.sdf'):
            os.system(f'cp /vols/opig/users/durant/pose_classification/conformer_random_walks/new_pdb_dataset_{i}/{pid}_{chain}/{pid}_{chain}_ligand_0.sdf {name}/{pid}/{code}_ligand_pbinvalid_{num}.sdf')
        else:
            if os.path.exists(f'/vols/opig/users/durant/pose_classification/conformer_random_walks/new_pdb_dataset_inter_pl_clashes/{pid}_{chain}/{pid}_{chain}_ligand_{num}.sdf'):
                os.system(f'cp /vols/opig/users/durant/pose_classification/conformer_random_walks/new_pdb_dataset_inter_pl_clashes/{pid}_{chain}/{pid}_{chain}_ligand_{num}.sdf {name}/{pid}/{code}_ligand_pbinvalid_{num}.sdf')
            else:
                if code not in problematic:
                    problematic.append(code)
      
        
    

100%|██████████| 1000/1000 [04:51<00:00,  3.43it/s]


In [13]:
print(len(problematic))

123


In [26]:
# make training df
# NEW_DF = pd.DataFrame(columns=['ligand', 'protein', 'label', 'pid'])
# ligand_files = []
# protein_files = []
# labels = []
# pids = []
# for code in tqdm(CODES):
#     pid = code.split('_')[0]
#     chain = code.split('_')[1]
#     res = code.split('_')[2]
#     if code in problematic:
#         continue
#     for i in range(0, 5):
#         ligand_files.append(f'{name}/{pid}/{code}_ligand_{i}.sdf')
#         protein_files.append(f'{name}/{pid}/{pid}_protein_cleaned.pdb')
#         labels.append(1)
#         pids.append(pid)
#     for i in range(0, 4):
#         ligand_files.append(f'{name}/{pid}/{code}_ligand_pbinvalid_{i}.sdf')
#         protein_files.append(f'{name}/{pid}/{pid}_protein_cleaned.pdb')
#         labels.append(0)
#         pids.append(pid)
# NEW_DF['ligand'] = ligand_files
# NEW_DF['protein'] = protein_files
# NEW_DF['label'] = labels
# NEW_DF['pid'] = pids    
# TRAIN_DF = NEW_DF.sample(frac=0.8)
# TEST_DF = NEW_DF.drop(TRAIN_DF.index)
# TRAIN_DF.to_csv(f'train_debug.csv', index=False)
# TEST_DF.to_csv(f'val_debug.csv', index=False)
        

100%|██████████| 1000/1000 [00:00<00:00, 53194.13it/s]

In [23]:
decoy_problematic = []

for code in tqdm(CODES):
    pid = code.split('_')[0]
    chain = code.split('_')[1]
    res = code.split('_')[2]
    if not os.path.exists(f'{name}/{pid}'):
        os.makedirs(f'{name}/{pid}')
    count = 0
    if not os.path.exists(f'/vols/opig/users/durant/pose_classification/data_prep/new_pdb_dataset/2024_06_19_after_structure_filters_complexes/{pid}/{pid}_ligand_{chain}_{res}.sdf'):
        decoy_problematic.append(code)
        continue
    true_mol = Chem.MolFromMolFile(f'/vols/opig/users/durant/pose_classification/data_prep/new_pdb_dataset/2024_06_19_after_structure_filters_complexes/{pid}/{pid}_ligand_{chain}_{res}.sdf')
    for i in range(50):
        if count >= 5:
            break
        if os.path.exists(f'/vols/opig/users/durant/pose_classification/conformer_random_walks/new_pdb_dataset_decoys/{pid}_{chain}_{i}.sdf/{pid}_{chain}_{num}.sdf_ligand_0.sdf'):
            test_mol = Chem.MolFromMolFile(f'/vols/opig/users/durant/pose_classification/conformer_random_walks/new_pdb_dataset_decoys/{pid}_{chain}_{i}.sdf/{pid}_{chain}_{num}.sdf_ligand_0.sdf')
            rmsd = check_rmsd(true_mol, test_mol)['results']['rmsd']
            print(rmsd)
        
    

  2%|▏         | 15/1000 [00:00<00:07, 124.74it/s]

18.741690164878825


  3%|▎         | 28/1000 [00:00<00:16, 60.06it/s] 

8.556242779446674
13.980616882174902
17.363029966902737


  4%|▎         | 36/1000 [00:00<00:18, 52.90it/s]

...

 99%|█████████▉| 988/1000 [04:50<00:00, 35.60it/s]

7.796779954738283
9.106374452459113
9.87670879354555
20.346964275038594


100%|█████████▉| 996/1000 [04:51<00:00, 33.97it/s]

7.977413208876139
9.29198379612387


100%|██████████| 1000/1000 [04:51<00:00,  3.43it/s]
